# Chapter 32 Guided To-Do Practice Labs

**Book Reference:** Chapter 32 (Part VI Capstone), Section 32.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 native PyTorch Transformer and LoRA building blocks, and pass each **function or class name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch32_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q sharukh-tutor
import math
import numpy as np
import torch
import torch.nn as nn
import sharukh_tutor as tutor

tutor.activate(chapter="ch32", notebook="guided_todos")


---
## To-Do 32.1 (Core Attention): `scaled_dot_product_attention(Q, K, V, mask=None)`

Implement `scaled_dot_product_attention(Q, K, V, mask=None)` supporting both 3D `(B, T, d_k)` and 4D `(B, H, T, d_k)` tensors:
- Compute `scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))`.
- If `mask is not None`, fill blocked positions (`mask == 0`) with `float("-inf")`.
- Compute `attn = torch.softmax(scores, dim=-1)` and return `(attn @ V, attn)`.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention (mask: 1=allow, 0=block)."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    d_k = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    return attn @ V, attn
    ### END SOLUTION


In [ ]:
tutor.check("ch32_todo_01", scaled_dot_product_attention)


---
## To-Do 32.2 (Sequence Order): `sinusoidal_positional_encoding(max_len, d_model)`

Implement Vaswani et al. (2017) sinusoidal positional encoding returning a `(max_len, d_model)` `torch.float32` tensor:
- Even columns `0::2`: $\sin(\text{pos} / 10000^{2i / d_{\text{model}}})$.
- Odd columns `1::2`: $\cos(\text{pos} / 10000^{2i / d_{\text{model}}})$.

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """Return (max_len, d_model) float32 sinusoidal position table."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    pe = torch.zeros(max_len, d_model, dtype=torch.float32)
    pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, d_model, 2, dtype=torch.float32)
        * (-math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(pos * div_term)
    pe[:, 1::2] = torch.cos(pos * div_term[: d_model // 2])
    return pe
    ### END SOLUTION


In [ ]:
tutor.check("ch32_todo_02", sinusoidal_positional_encoding)


---
## To-Do 32.3 (Parallel Subspaces): `MultiHeadSelfAttention`

Build `class MultiHeadSelfAttention(nn.Module)`:
- `__init__(self, d_model, num_heads)`: validate `d_model % num_heads == 0` (raising `ValueError` otherwise) and create `W_q`, `W_k`, `W_v`, `W_o` as `nn.Linear(d_model, d_model)`.
- `forward(self, x, mask=None)`: project and reshape to `(B, H, T, d_k)`, call `scaled_dot_product_attention`, recombine heads into `(B, T, d_model)`, project with `W_o`, and return `(out, attn_weights)`.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Multi-head self-attention with W_q, W_k, W_v, W_o projections."""

    def __init__(self, d_model, num_heads):
        super().__init__()
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        if d_model % num_heads != 0:
            raise ValueError(
                "d_model must be divisible by num_heads"
            )
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        ### END SOLUTION

    def forward(self, x, mask=None):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        B, T, _ = x.shape
        H, dk = self.num_heads, self.d_k
        Q = self.W_q(x).view(B, T, H, dk).transpose(1, 2)
        K = self.W_k(x).view(B, T, H, dk).transpose(1, 2)
        V = self.W_v(x).view(B, T, H, dk).transpose(1, 2)
        ctx, attn = scaled_dot_product_attention(Q, K, V, mask=mask)
        merged = ctx.transpose(1, 2).contiguous().view(
            B, T, self.d_model
        )
        return self.W_o(merged), attn
        ### END SOLUTION


In [ ]:
tutor.check("ch32_todo_03", MultiHeadSelfAttention)


---
## To-Do 32.4 (Transformer Layer): `TransformerEncoderBlock`

Build `class TransformerEncoderBlock(nn.Module)` with `__init__(self, d_model, num_heads, d_ff, dropout=0.1)` combining:
- `MultiHeadSelfAttention(d_model, num_heads)` + residual connection + `nn.LayerNorm(d_model)`.
- 2-layer feed-forward network (`Linear(d_model, d_ff) -> GELU -> Dropout -> Linear(d_ff, d_model)`) + residual connection + second `nn.LayerNorm(d_model)`.
- `forward(self, x, mask=None)` returns `(out[B, T, d_model], attn_weights)`.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Transformer Encoder Block: MHA + LayerNorm + GELU FFN."""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        ### END SOLUTION

    def forward(self, x, mask=None):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        attn_out, weights = self.attn(x, mask=mask)
        x = self.norm1(x + self.drop(attn_out))
        x = self.norm2(x + self.drop(self.ffn(x)))
        return x, weights
        ### END SOLUTION


In [ ]:
tutor.check("ch32_todo_04", TransformerEncoderBlock)


---
## To-Do 32.5 (Text Pipeline): `ClinicalVocabTokenizer`

Build `class ClinicalVocabTokenizer` with:
- Class constants `PAD_ID = 0` and `UNK_ID = 1`.
- `.fit(self, texts)`: lowercase & split whitespace into a deterministic sorted vocabulary starting at ID `2`, returning `self`.
- `.encode(self, text, max_len)`: return `list[int]` of exact length `max_len`, mapping OOV tokens to `UNK_ID` and right-padding with `PAD_ID`.
- `__len__(self)`: return total vocabulary size.

In [ ]:
class ClinicalVocabTokenizer:
    """Deterministic whitespace clinical tokenizer with PAD & UNK."""

    PAD_ID = 0
    UNK_ID = 1

    def __init__(self):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        self.token_to_id = {"<PAD>": 0, "<UNK>": 1}
        self.id_to_token = {0: "<PAD>", 1: "<UNK>"}
        ### END SOLUTION

    def fit(self, texts):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        words = sorted(
            {w for t in texts for w in t.lower().split()}
        )
        self.token_to_id = {"<PAD>": 0, "<UNK>": 1}
        for idx, word in enumerate(words, start=2):
            self.token_to_id[word] = idx
        self.id_to_token = {
            v: k for k, v in self.token_to_id.items()
        }
        return self
        ### END SOLUTION

    def encode(self, text, max_len):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        ids = [
            self.token_to_id.get(w, self.UNK_ID)
            for w in text.lower().split()
        ][:max_len]
        return ids + [self.PAD_ID] * (max_len - len(ids))
        ### END SOLUTION

    def __len__(self):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        return len(self.token_to_id)
        ### END SOLUTION


In [ ]:
tutor.check("ch32_todo_05", ClinicalVocabTokenizer)


---
## To-Do 32.6 (PEFT Adapter): `LoRALinear`

Build `class LoRALinear(nn.Module)` with `__init__(self, base_linear, rank=4, alpha=8.0)`:
- Freeze all parameters of `base_linear` (`requires_grad = False`).
- Create trainable `self.lora_A = nn.Parameter(...)` of shape `(rank, in_features)` with small random init and `self.lora_B = nn.Parameter(torch.zeros(out_features, rank))`.
- Compute `base_linear(x) + (alpha / rank) * (x @ lora_A.T @ lora_B.T)` so step-0 output matches `base_linear(x)` identically.

In [ ]:
class LoRALinear(nn.Module):
    """Freeze base nn.Linear and learn low-rank delta (B @ A)."""

    def __init__(self, base_linear, rank=4, alpha=8.0):
        super().__init__()
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        self.base_linear = base_linear
        for p in self.base_linear.parameters():
            p.requires_grad = False
        self.rank = rank
        self.alpha = float(alpha)
        self.scaling = float(alpha) / rank
        in_f = base_linear.in_features
        out_f = base_linear.out_features
        self.lora_A = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(out_f, rank))
        ### END SOLUTION

    def forward(self, x):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        base_out = self.base_linear(x)
        delta = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + delta
        ### END SOLUTION


In [ ]:
tutor.check("ch32_todo_06", LoRALinear)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 32.1 (Missing `1 / sqrt(d_k)` Scaling or Post-Softmax Masking):** Omitting `/ math.sqrt(d_k)` drives large dot products into softmax saturation where gradients vanish. Likewise, multiplying weights by `mask` *after* softmax breaks probability normalization—always call `scores.masked_fill(mask == 0, float("-inf"))` *before* softmax.
- **To-Do 32.3 (Head Reshape Ordering Bug):** Calling `.view(B, H, T, d_k)` directly on `(B, T, d_model)` scrambles adjacent tokens across heads! Always split the last dimension first (`view(B, T, H, d_k)`), swap axes (`transpose(1, 2)`), and reverse cleanly with `.transpose(1, 2).contiguous().view(B, T, d_model)`.
- **To-Do 32.6 (Zero-Initializing `lora_B` vs. `lora_A`):** Initializing `lora_B = 0` while `lora_A` is small Gaussian noise guarantees identical step-0 outputs to the frozen pretrained model *and* non-zero first-step gradients on `lora_B`. Initializing *both* `A` and `B` to zero creates a dead saddle point where neither matrix learns!

In [ ]:
tutor.summary()
